# TrOCR V2: ochrona zwyklego druku

Wybierz **GPU T4** i **Uruchom wszystko**. Nie wgrywaj zadnych plikow.

Trzy warianty po 3 epoki: LR 1e-5 / replay 500, LR 3e-6 / replay 500, LR 3e-6 / replay 2000. Wszystkie zaczynaja od tej samej przypietej bazy. 70 linii historycznych w treningu; kontrola: 9 historycznych i 75 zwyklych.

Kandydat musi poprawic historyczny i laczny CER, bez regresji CER/WER zwyklego druku i bez dodatkowych znakow zastepczych. W przeciwnym razie pozostaje baza. Stara pisownia pozostaje bez zmian. To wybor na zbiorze development, nie dowod SOTA.

Raport pobierze sie automatycznie. Dla duzego ZIP-a z wagami jest opcjonalny zapis na Dysku Google w ostatniej komorce. Nie wymaga to ponownego treningu.

## 1. Srodowisko

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import venv

CODE_REVISION = 'ee90eb4426232f0a03937787c98291b59ce8398a'
DATA_REVISION = '3a301fd12a6272b28774d1888213ddf528b47f24'
DATA_SHA256 = '3e39b7d02b90a0a741adde506c0467698c46fe86c82846d3cc0e648eebaae32e'
repo = Path('/content/OCR_engine-reviewed-v2')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/PiotrStyla/OCR_engine.git', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'fetch', 'origin', 'main'], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '--detach', CODE_REVISION], check=True)
assert subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip() == CODE_REVISION
environment = Path('/content/slayer-reviewed-training-v2-env')
python = environment/'bin/python'
if not python.exists():
    venv.EnvBuilder(with_pip=False, system_site_packages=True).create(environment)
subprocess.run([str(python), '-c', "import torch; assert torch.cuda.is_available(), 'Wybierz GPU T4 w ustawieniach Colaba i uruchom wszystko.'; print('GPU:', torch.cuda.get_device_name(0))"], check=True)
torch_release = subprocess.check_output([str(python), '-c', "import torch; print('.'.join(torch.__version__.split('+')[0].split('.')[:2]))"], text=True).strip()
torchao_versions = {'2.10': '0.16.0', '2.11': '0.17.0'}
assert torch_release in torchao_versions, f'Niezweryfikowana wersja Torch: {torch_release}. Nie zmieniam CUDA.'
packages = ['transformers==4.57.6', 'tokenizers==0.22.2', 'huggingface_hub==0.36.2',
            'peft==0.19.1', 'accelerate==1.13.0', 'jiwer==4.0.0',
            'pillow==11.3.0', 'sentencepiece==0.2.1', 'pytest==8.4.2']
subprocess.run([str(python), '-m', 'pip', 'install', '--no-cache-dir', *packages], check=True)
subprocess.run([str(python), '-m', 'pip', 'install', '--no-cache-dir', '--no-deps',
                'torchao=='+torchao_versions[torch_release]], check=True)
subprocess.run([str(python), '-c', "from transformers import TrOCRProcessor, VisionEncoderDecoderModel; from transformers.generation import GenerationMixin; import peft, jiwer; print('TRAINING_IMPORTS_OK')"], check=True)
subprocess.run([str(python), '-m', 'training.preflight_reviewed_recognizer',
                '--output', str(environment/'adapter-preflight.json')], cwd=repo, check=True)
print('Gotowe. Dane zostana pobrane automatycznie; niczego nie wgrywaj.')
os.environ['PYTEST_DISABLE_PLUGIN_AUTOLOAD'] = '1'
subprocess.run([str(python), '-m', 'pytest', 'tests/test_training_protocol.py',
                '-q'], cwd=repo, check=True)
os.environ['HF_HUB_DISABLE_XET'] = '1'


## 2. Dane, kontrola, trening i porownanie

In [ ]:
from google.colab import files

RESULT_DIRECTORY = None
EVIDENCE_ZIP = None
command = [str(python), '-m', 'training.run_reviewed_recognizer_colab_v2']
process = subprocess.Popen(command, cwd=repo, stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end='', flush=True)
    if line.startswith('RESULT_DIRECTORY '):
        RESULT_DIRECTORY = Path(line.strip().split(' ', 1)[1])
    elif line.startswith('EVIDENCE_ZIP '):
        EVIDENCE_ZIP = Path(line.strip().split(' ', 1)[1])
return_code = process.wait()
if return_code:
    if EVIDENCE_ZIP is not None and EVIDENCE_ZIP.is_file():
        files.download(str(EVIDENCE_ZIP))
    raise RuntimeError('Trening nie zakonczyl sie. Zapisano ZIP diagnostyczny; nie jest to model.')
assert RESULT_DIRECTORY is not None, 'Brak potwierdzenia zakonczenia treningu.'
print('TRENING_ZAKONCZONY', RESULT_DIRECTORY)


## 3. Pobranie modelu i raportu

In [ ]:
assert RESULT_DIRECTORY is not None, 'Najpierw musi zakonczyc sie trening.'
assert EVIDENCE_ZIP is not None and EVIDENCE_ZIP.is_file(), 'Brak raportu.'
result_zip = RESULT_DIRECTORY/'recognizer-reviewed-colab-v2-result.zip'
assert result_zip.is_file(), 'Brak kompletnego archiwum wyniku.'
import json
selection = json.loads((RESULT_DIRECTORY/'evidence/selection.json').read_text())
print('Wybor:', selection['selected'])
print('CER: baza i warianty (historyczny / zwykly / laczny)')
baseline = json.loads((RESULT_DIRECTORY/'evidence/baseline-metrics.json').read_text())
for name, metrics in [('unchanged-baseline', baseline)] + [(row['id'], row['metrics']) for row in selection['candidates']]:
    print(name, ' / '.join(f"{100*metrics[domain]['cer']:.3f}%" for domain in
          ('historical-development', 'ordinary-development', 'combined')))
for row in selection['candidates']:
    print(row['id'], 'przechodzi' if row['eligible'] else 'odrzucony: '+', '.join(row['reasons']))
print('Kompletny pakiet:', result_zip, 'Bajty:', result_zip.stat().st_size)
print('Jesli pozostaje baza, ZIP zawiera raport i przypieta tozsamosc bazy, bez nowych wag.')
files.download(str(EVIDENCE_ZIP))


## 4. Zachowaj pelny pakiet na Drive (opcjonalnie)

In [ ]:
BACKUP_TO_DRIVE = False
# Aby zachowac duzy pakiet, zmien powyzej na True i uruchom tylko te komorke.
if BACKUP_TO_DRIVE:
    from google.colab import drive
    import hashlib
    import shutil
    drive.mount('/content/drive')
    destination = Path('/content/drive/MyDrive/OCR_engine')/RESULT_DIRECTORY.name
    destination.mkdir(parents=True, exist_ok=True)
    target = destination/result_zip.name
    def sha256(path):
        h = hashlib.sha256()
        with path.open('rb') as stream:
            for chunk in iter(lambda: stream.read(8*1024*1024), b''):
                h.update(chunk)
        return h.hexdigest()
    expected = sha256(result_zip)
    if target.exists():
        assert sha256(target) == expected, 'Na Drive istnieje inny plik; nie nadpisuje go.'
    else:
        shutil.copyfile(result_zip, target)
    assert sha256(target) == expected, 'Kopia na Drive nie przeszla kontroli SHA256.'
    print('Zapisano kompletny pakiet na Drive:', target, 'SHA256:', expected)
else:
    print('Wagi sa w sesji Colaba. Aby zachowac je przed jej zakonczeniem, ustaw BACKUP_TO_DRIVE=True i uruchom tylko te komorke.')
